In [ ]:
import pandas as pd
import altair as alt
import scipy.stats as stats

In [ ]:
@alt.theme.register('arial_theme', enable=True)
def arial_theme():
    return alt.theme.ThemeConfig({
        'config': {
            'font': 'Arial',
            'title': {'font': 'Arial'},
            'axis': {'labelFont': 'Arial', 'titleFont': 'Arial'},
            'legend': {'labelFont': 'Arial', 'titleFont': 'Arial'},
            'header': {'labelFont': 'Arial', 'titleFont': 'Arial'},
            'text': {'font': 'Arial'},
        }
    })

In [ ]:
lib_a_oligos =pd.read_csv('/Users/ivan/Documents/GitHub/BARD1_SGE_analysis/Data/extra_data/20260603_BARD1_X4A_IDR_LibA.csv')
lib_b_oligos=pd.read_csv('/Users/ivan/Documents/GitHub/BARD1_SGE_analysis/Data/extra_data/20260603_BARD1_X4A_IDR_LibB.csv')

lib_a_lfcs=pd.read_csv('/Users/ivan/Documents/GitHub/BARD1_SGE_analysis/Data/extra_data/20260812_BARD1_X4A_IDR_LibA_LFCs.tsv', sep='\t')
lib_b_lfcs=pd.read_csv('/Users/ivan/Documents/GitHub/BARD1_SGE_analysis/Data/extra_data/20260812_BARD1_X4A_IDR_LibB_LFCs.tsv', sep='\t')

full_sat_fitness_scores=pd.read_csv('/Users/ivan/Documents/GitHub/BARD1_SGE_analysis/Data/extra_data/20260828_BARD1_X4A_IDR_LibA_aminoacidsat_fitness_scores.tsv', sep='\t')

# Analyze Library A

In [ ]:
lib_a_merged = pd.merge(lib_a_lfcs, lib_a_oligos, left_on='oligo_name', right_on='seq_name', how='inner')

lib_a_merged

In [ ]:
## Validate that name-matched rows also agree on sequence.
# lib_a_oligos['seq'] is the full designed oligo (AMP_F + edited region + AMP_R,
# sense strand). lib_a_lfcs['oligo_seq'] is only the edited core region, read off
# the antisense strand. So to confirm the merge isn't just matching names while
# silently pairing unrelated sequences, trim the primers off 'seq' and compare
# to the reverse complement of 'oligo_seq'. See SGE_BARD1_X4_IDR_libs.ipynb for
# BARD1_X4a_AMP_F / BARD1_X4a_AMP_R and how the oligos were built.

BARD1_X4a_AMP_F = 'CCATGTGGGAGCAATAAATTTC'
BARD1_X4a_AMP_R = 'CCCTCGAAGTAAGAAAGTCAG'

def revcomp(seq):
    comp = str.maketrans('ACGTacgt', 'TGCAtgca')
    return seq.translate(comp)[::-1]

def core_region(seq):
    seq = seq.upper()
    assert seq.startswith(BARD1_X4a_AMP_F), f'missing AMP_F: {seq[:30]}'
    assert seq.endswith(BARD1_X4a_AMP_R), f'missing AMP_R: {seq[-30:]}'
    return seq[len(BARD1_X4a_AMP_F):len(seq) - len(BARD1_X4a_AMP_R)]

seq_matches = lib_a_merged['seq'].apply(core_region) == lib_a_merged['oligo_seq'].apply(revcomp).str.upper()

assert seq_matches.all(), f'{(~seq_matches).sum()} of {len(lib_a_merged)} merged rows have mismatched sequences'
print(f'All {len(lib_a_merged)} merged rows agree on sequence (name match backed by sequence match).')

lib_a_final=lib_a_merged[['oligo_name', 'oligo_seq', 'var_name', 'D13_log2_median']].copy()

lib_a_final

## 6-bp Deletions & ClinVar Indels

In [ ]:
del_df = lib_a_final[lib_a_final['oligo_name'].str.contains('6bp_del')].copy()
del_df['start'] = del_df['var_name'].transform(lambda x: int(x.split('-')[0]))
del_df['end'] = del_df['var_name'].transform(lambda x: int(x.split('-')[1]))

In [ ]:
del_df['variant_type'] = '6-bp deletion'

# ClinVar delins (see SGE_BARD1_X4_IDR_libs.ipynb, ## ClinVar Variants). The
# code there builds them in ascending CDS-coordinate order, which is the
# reverse of how the markdown there lists them by ClinVar entry:
#   ClinVarIndels_1: c.377_378delinsGA          (p.Asp126Gly)
#   ClinVarIndels_2: c.403_404delinsTT          (p.Asp135Phe)
#   ClinVarIndels_3: c.420_421delinsTT          (p.Lys140_Asn141delinsAsnTyr)
#   ClinVarIndels_4: c.433_435delinsTA          (p.Met145fs) -- a real
#     frameshift (3 bases replaced by 2), unlike 1-3 which are same-length
#     substitutions (2-for-2).
#   ClinVarIndels_5: c.365-20_365-1delinsAA     -- intronic/splice-region,
#     dropped out during sequencing, no score.
# None of these are in lib_b_lfcs, so the 4 available scores are supplied
# directly. Genomic coordinates are recovered the same way as the 6-bp dels:
# c_to_ref() maps a CDS position to an index in x4a_upstream + x4a_to_mutate +
# x4a_downstream, and x4a_to_mutate sits on the antisense strand (its first
# base is at mut_end_coord, coordinate decreasing as index increases).
mut_end_coord = 214781507  # from SGE_BARD1_X4_IDR_libs.ipynb
CDS_C_START = 367

def clinvar_genomic_coord(c_pos):
    return mut_end_coord - (c_pos - CDS_C_START)

clinvar_indel_spans = [
    ('BARD1_X4A_IDR_ClinVarIndels_1', 377, 378),
    ('BARD1_X4A_IDR_ClinVarIndels_2', 403, 404),
    ('BARD1_X4A_IDR_ClinVarIndels_3', 420, 421),
    ('BARD1_X4A_IDR_ClinVarIndels_4', 433, 435),
]
clinvar_scores = {
    'BARD1_X4A_IDR_ClinVarIndels_1': 0.634241,
    'BARD1_X4A_IDR_ClinVarIndels_2': -0.100195,
    'BARD1_X4A_IDR_ClinVarIndels_3': -0.405805,
    'BARD1_X4A_IDR_ClinVarIndels_4': -4.793020,
}

clinvar_rows = []
for name, c_start, c_end in clinvar_indel_spans:
    g_start = clinvar_genomic_coord(c_start)
    g_end = clinvar_genomic_coord(c_end)
    clinvar_rows.append({
        'oligo_name': name,
        'start': min(g_start, g_end),
        'end': max(g_start, g_end),
        'D13_log2_median': clinvar_scores[name],
        'variant_type': 'ClinVar indel',
    })

clinvar_df = pd.DataFrame(clinvar_rows)

combined_del_df = pd.concat([
    del_df[['oligo_name', 'start', 'end', 'D13_log2_median', 'variant_type']],
    clinvar_df[['oligo_name', 'start', 'end', 'D13_log2_median', 'variant_type']],
], ignore_index=True)

strip_plot = alt.Chart(combined_del_df).mark_line(strokeWidth=3).encode(
    x=alt.X('start:Q',
            scale=alt.Scale(zero=False),
            title='Genomic position'
    ),
    x2='end:Q',
    y=alt.Y('D13_log2_median:Q', title='D13 log2 median LFC'),
    color=alt.Color(
        'variant_type:N',
        scale=alt.Scale(domain=['6-bp deletion', 'ClinVar indel'], range=['#2a78d6', '#eb6834']),
        legend=alt.Legend(title=None)
    ),
    tooltip=['oligo_name:N', 'start:Q', 'end:Q', alt.Tooltip('D13_log2_median:Q', format='.2f')]
).properties(
    width=800,
    height=150
)

strip_plot.display()

## Lys to Ala 

In [ ]:
lysala_df=lib_a_final[lib_a_final['oligo_name'].str.contains('LystoAla')].copy()
lysala_df

In [ ]:
combined_del_df

In [ ]:
import re

position_order = [124, 127, 130, 139, 140, 144]

lysala_df['positions'] = lysala_df['var_name'].apply(lambda v: [int(p) for p in re.findall(r'K(\d+)', v)])
lysala_df['n_mutations'] = lysala_df['positions'].apply(len)

lysala_long = lysala_df.explode('positions').rename(columns={'positions': 'position'})
lysala_long['position'] = lysala_long['position'].astype(int)

lysala_long

In [ ]:
BLUE = '#2a78d6'
MUTED = '#898781'
RED = '#e34948'
DEPLETION_THRESHOLD = -0.5

# "Hotness" = of all 32 variants that include a given position, what fraction
# score below the depletion threshold. Every position appears in exactly 32 of
# the 63 combos, so raw depleted-counts are directly comparable across positions.
depleted = lysala_long[lysala_long['D13_log2_median'] < DEPLETION_THRESHOLD]
hotness = (
    depleted.groupby('position')['var_name'].nunique()
    / lysala_long.groupby('position')['var_name'].nunique()
).reindex(position_order).rename('depleted_frac').reset_index()

hot_bg = alt.Chart(hotness).mark_rect().encode(
    x=alt.X('position:O', sort=position_order),
    opacity=alt.Opacity(
        'depleted_frac:Q',
        scale=alt.Scale(range=[0, 0.35]),
        legend=alt.Legend(title='Frac. depleted (<-0.5)', format='.0%')
    ),
    color=alt.value(RED)
)

lines = alt.Chart().mark_line(
    color=BLUE,
    strokeWidth=2,
    point=alt.OverlayMarkDef(color=BLUE, filled=True, size=70, stroke='white', strokeWidth=2)
).encode(
    x=alt.X('position:O', sort=position_order, title='Mutated Lys position'),
    y=alt.Y('D13_log2_median:Q', title='D13 log2 median LFC', scale=alt.Scale(zero=False, padding=10)),
    detail='var_name:N',
    tooltip=['var_name:N', alt.Tooltip('D13_log2_median:Q', format='.2f'), 'n_mutations:O']
)

ref_line = alt.Chart(pd.DataFrame({'y': [DEPLETION_THRESHOLD]})).mark_rule(
    color=MUTED, strokeWidth=1, strokeDash=[4, 4]
).encode(
    y='y:Q'
)

lysala_barbell = alt.layer(hot_bg, lines, ref_line, data=lysala_long).properties(
    width=220, height=260
).facet(
    facet=alt.Facet('n_mutations:O', title='# Lys→Ala mutations'),
    columns=3
).resolve_scale(
    y='independent'
).properties(
    title='Lys→Ala combinations: D13 log2 median fitness by mutated residue'
).configure(
    background='#fcfcfb'
).configure_axis(
    gridColor='#e1e0d9', domainColor='#c3c2b7', labelColor='#52514e', titleColor='#0b0b0b'
).configure_view(
    strokeWidth=0
).configure_title(
    color='#0b0b0b'
)

lysala_barbell

### Focus view: zoomed to scores < -0.4, depletion threshold stays -0.5

In [ ]:
DEPLETION_THRESHOLD = -0.5
ZOOM_MAX = -0.4

# Global hotness (same metric as the main chart): of all 32 variants containing
# a given position, what fraction score below DEPLETION_THRESHOLD. Every
# position appears in exactly 32 of the 63 combos, so counts are comparable.
depleted = lysala_long[lysala_long['D13_log2_median'] < DEPLETION_THRESHOLD]
hotness = (
    depleted.groupby('position')['var_name'].nunique()
    / lysala_long.groupby('position')['var_name'].nunique()
).reindex(position_order).rename('depleted_frac').reset_index()

# Zoom: only display variants scoring below ZOOM_MAX (broader than the -0.5
# depletion cutoff, so near-threshold variants stay visible for context).
lysala_focus_long = lysala_long[lysala_long['D13_log2_median'] < ZOOM_MAX].copy()

hot_bg_focus = alt.Chart(hotness).mark_rect().encode(
    x=alt.X('position:O', sort=position_order),
    opacity=alt.Opacity(
        'depleted_frac:Q',
        scale=alt.Scale(range=[0, 0.35]),
        legend=alt.Legend(title='Frac. depleted (<-0.5)', format='.0%')
    ),
    color=alt.value(RED)
)

lines_focus = alt.Chart().mark_line(
    color=BLUE,
    strokeWidth=2,
    point=alt.OverlayMarkDef(color=BLUE, filled=True, size=70, stroke='white', strokeWidth=2)
).encode(
    x=alt.X('position:O', sort=position_order, title='Mutated Lys position'),
    y=alt.Y('D13_log2_median:Q', title='D13 log2 median LFC', scale=alt.Scale(zero=False, padding=10)),
    detail='var_name:N',
    tooltip=['var_name:N', alt.Tooltip('D13_log2_median:Q', format='.2f'), 'n_mutations:O']
)

ref_line_focus = alt.Chart(pd.DataFrame({'y': [DEPLETION_THRESHOLD]})).mark_rule(
    color=MUTED, strokeWidth=1, strokeDash=[4, 4]
).encode(
    y='y:Q'
)

lysala_barbell_focus = alt.layer(hot_bg_focus, lines_focus, ref_line_focus, data=lysala_focus_long).properties(
    width=220, height=260
).facet(
    facet=alt.Facet('n_mutations:O', title='# Lys→Ala mutations'),
    columns=3
).resolve_scale(
    y='independent'
).properties(
    title='Lys→Ala combinations: D13 log2 median fitness by mutated residue'
).configure(
    background='#fcfcfb'
).configure_axis(
    gridColor='#e1e0d9', domainColor='#c3c2b7', labelColor='#52514e', titleColor='#0b0b0b'
).configure_view(
    strokeWidth=0
).configure_title(
    color='#0b0b0b'
)

lysala_barbell_focus

### Positional co-occurrence: which positions travel together?

Two questions:
1. Among depleted variants, and separately among non-depleted variants, are
   certain positions frequently found together?
2. At hot positions like K140/K144, not every variant depletes — among the
   non-depleting variants that do include K140 or K144, is there a particular
   position they're frequently found with?

For each group (depleted / non-depleted), compute P(column position mutated |
row position mutated) — conditioning on the row position controls for the fact
that hot positions simply appear more often, so this isolates genuine pairing
tendency rather than just marginal frequency.

In [ ]:
import itertools

def cooccurrence_matrix(df):
    mat = pd.DataFrame(0.0, index=position_order, columns=position_order)
    for positions in df['positions']:
        for a, b in itertools.combinations(positions, 2):
            mat.loc[a, b] += 1
            mat.loc[b, a] += 1
        for a in positions:
            mat.loc[a, a] += 1
    return mat

lysala_df['depleted'] = lysala_df['D13_log2_median'] < DEPLETION_THRESHOLD

cooc_rows = []
for group_label, sub in [
    ('Depleted (< -0.5)', lysala_df[lysala_df['depleted']]),
    ('Non-depleted (>= -0.5)', lysala_df[~lysala_df['depleted']]),
]:
    mat = cooccurrence_matrix(sub)
    diag = pd.Series(mat.values.diagonal(), index=mat.index)
    cond = mat.div(diag, axis=0)  # P(other | given), row-normalized
    for i in position_order:
        for j in position_order:
            if i == j:
                continue
            cooc_rows.append({
                'group': group_label,
                'given_position': i,
                'other_position': j,
                'cond_prob': cond.loc[i, j],
                'n_given': int(diag[i]),
            })

cooc_long = pd.DataFrame(cooc_rows)
cooc_long

In [ ]:
BLUE_RAMP = ['#cde2fb', '#9ec5f4', '#5598e7', '#2a78d6', '#184f95']

cooc_heat = alt.Chart(cooc_long).mark_rect().encode(
    x=alt.X('other_position:O', sort=position_order, title='Co-mutated position'),
    y=alt.Y('given_position:O', sort=position_order, title='Given this position is mutated'),
    color=alt.Color(
        'cond_prob:Q',
        scale=alt.Scale(range=BLUE_RAMP, domain=[0, 1]),
        legend=alt.Legend(title='P(other | given)', format='.0%')
    ),
    tooltip=['given_position:O', 'other_position:O', alt.Tooltip('cond_prob:Q', format='.0%'), 'n_given:Q']
)

cooc_text = alt.Chart(cooc_long).mark_text(fontSize=11).encode(
    x=alt.X('other_position:O', sort=position_order),
    y=alt.Y('given_position:O', sort=position_order),
    text=alt.Text('cond_prob:Q', format='.0%'),
    color=alt.condition('datum.cond_prob > 0.55', alt.value('white'), alt.value('#0b0b0b'))
)

cooc_chart = alt.layer(cooc_heat, cooc_text).properties(width=200, height=200).facet(
    facet=alt.Facet('group:N', title=None),
    columns=2
).properties(
    title='Positional co-occurrence: P(other position mutated | given position mutated)'
).configure(
    background='#fcfcfb'
).configure_axis(
    labelColor='#52514e', titleColor='#0b0b0b', domainColor='#c3c2b7'
).configure_view(
    strokeWidth=0
).configure_title(
    color='#0b0b0b'
)

cooc_chart

**Reading the two panels:**

- **Depleted variants** (left): no column stands out — probabilities are fairly
  uniform (~50-75%) across every row. Depletion looks driven mainly by *how
  many* positions are mutated (see the faceted views above), not by a specific
  pairing.
- **Non-depleted variants** (right): K127 is the standout column in almost
  every row — including 73% for K140 and 50% for K144, both well above that
  row's other partners. Among the survivors that still carry a hot position
  like K140 or K144, K127 is disproportionately likely to be riding along —
  consistent with K127 being individually the most fitness-neutral-to-positive
  single mutation, and hinting it may offset/rescue the cost of K140 or K144.

Caveat: the K140 and K144 rows in the non-depleted panel are based on only
11 and 10 variants respectively (32 combos containing that position, minus
however many depleted) — real signal, but a small-N one.

### Median score by number of positions changed

In [ ]:
median_by_n = lysala_df.groupby('n_mutations', as_index=False)['D13_log2_median'].median()

median_bars = alt.Chart(median_by_n).mark_bar(color=BLUE, size=24).encode(
    x=alt.X('n_mutations:O', title='# Lys→Ala mutations'),
    y=alt.Y('D13_log2_median:Q', title='Median D13 log2 median LFC')
)

median_labels_above = alt.Chart(median_by_n[median_by_n['D13_log2_median'] >= 0]).mark_text(
    dy=-8, color='#0b0b0b', fontSize=11
).encode(
    x=alt.X('n_mutations:O'), y=alt.Y('D13_log2_median:Q'), text=alt.Text('D13_log2_median:Q', format='.2f')
)
median_labels_below = alt.Chart(median_by_n[median_by_n['D13_log2_median'] < 0]).mark_text(
    dy=12, color='#0b0b0b', fontSize=11
).encode(
    x=alt.X('n_mutations:O'), y=alt.Y('D13_log2_median:Q'), text=alt.Text('D13_log2_median:Q', format='.2f')
)

median_ref = alt.Chart(pd.DataFrame({'y': [DEPLETION_THRESHOLD]})).mark_rule(
    color=MUTED, strokeWidth=1, strokeDash=[4, 4]
).encode(y='y:Q')

median_chart = (median_bars + median_labels_above + median_labels_below + median_ref).properties(
    width=400, height=280,
    title='Median D13 log2 LFC by number of Lys→Ala mutations'
).configure(
    background='#fcfcfb'
).configure_axis(
    gridColor='#e1e0d9', domainColor='#c3c2b7', labelColor='#52514e', titleColor='#0b0b0b'
).configure_view(
    strokeWidth=0
).configure_title(
    color='#0b0b0b'
)

median_chart

### Isolating each residue's individual contribution

The hotness plots answer "what fraction of combos containing this position are
depleted" — but that's still an average over whichever specific partners
happen to accompany the position across its 32 combos, so a naive read risks
attributing a position's effect to its typical company rather than to itself.

Because this library is a complete factorial (every non-empty subset of the 6
positions, balanced), we can isolate each position's contribution with a
simple additive model:

`score ~ intercept + sum_i (beta_i * [position i mutated])`

Since the design is balanced, `beta_i` is close to an unconfounded estimate of
position i's *average* marginal effect across the whole combinatorial
background — not just its effect in isolation. Comparing that to the raw
single-mutant score (zero confounding, but only tells you about the position
alone) shows how much of each position's apparent effect actually generalizes.

In [ ]:
import numpy as np

for p in position_order:
    lysala_df[f'x_{p}'] = lysala_df['positions'].apply(lambda ps, p=p: 1.0 if p in ps else 0.0)

X_cols = [f'x_{p}' for p in position_order]
X_design = np.column_stack([np.ones(len(lysala_df)), lysala_df[X_cols].values])
y = lysala_df['D13_log2_median'].values

beta, *_ = np.linalg.lstsq(X_design, y, rcond=None)
resid = y - X_design @ beta
dof = len(y) - X_design.shape[1]
sigma2 = (resid @ resid) / dof
se = np.sqrt(np.diag(np.linalg.inv(X_design.T @ X_design)) * sigma2)
r2 = 1 - (resid ** 2).sum() / ((y - y.mean()) ** 2).sum()

print(f'Additive model R^2 = {r2:.2f} (rest is epistasis/interaction + noise)')

single_mutant = lysala_df[lysala_df['n_mutations'] == 1].copy()
single_mutant['pos'] = single_mutant['positions'].apply(lambda x: x[0])
single_mutant = single_mutant.set_index('pos')['D13_log2_median'].reindex(position_order)

effects = pd.DataFrame({
    'position': position_order,
    'single_mutant': single_mutant.values,
    'main_effect': beta[1:],
    'main_effect_se': se[1:],
})
effects['main_effect_lo'] = effects['main_effect'] - 1.96 * effects['main_effect_se']
effects['main_effect_hi'] = effects['main_effect'] + 1.96 * effects['main_effect_se']

effects

In [ ]:
effects_long = effects.melt(
    id_vars=['position', 'main_effect_lo', 'main_effect_hi'],
    value_vars=['single_mutant', 'main_effect'],
    var_name='metric', value_name='value'
)

connector = alt.Chart(effects).mark_rule(color=MUTED, strokeWidth=1).encode(
    x=alt.X('position:O', sort=position_order),
    y='single_mutant:Q', y2='main_effect:Q'
)
main_effect_ci = alt.Chart(effects).mark_rule(color='#eb6834', strokeWidth=2).encode(
    x=alt.X('position:O', sort=position_order),
    y='main_effect_lo:Q', y2='main_effect_hi:Q'
)
effect_points = alt.Chart(effects_long).mark_point(filled=True, size=100).encode(
    x=alt.X('position:O', sort=position_order, title='Lys position'),
    y=alt.Y('value:Q', title='D13 log2 median LFC'),
    color=alt.Color(
        'metric:N',
        scale=alt.Scale(domain=['single_mutant', 'main_effect'], range=[BLUE, '#eb6834']),
        legend=alt.Legend(title=None)
    )
)
zero_ref = alt.Chart(pd.DataFrame({'y': [0]})).mark_rule(color=MUTED, strokeDash=[4, 4]).encode(y='y:Q')

effects_chart = (connector + main_effect_ci + effect_points + zero_ref).properties(
    width=420, height=320,
    title=f'Isolated single-mutant effect vs. average main effect (additive model R^2={r2:.2f})'
).configure(
    background='#fcfcfb'
).configure_axis(
    gridColor='#e1e0d9', domainColor='#c3c2b7', labelColor='#52514e', titleColor='#0b0b0b'
).configure_view(
    strokeWidth=0
).configure_title(
    color='#0b0b0b'
)

effects_chart

**Reading the chart:** for K139, K140, K144 the blue (isolated) and orange
(average main effect) dots sit close together, and the orange 95% CI clearly
excludes zero — these positions' negative effect is robust regardless of
genetic background. For K124, K127, K130 there's a large gap: strongly
positive alone (+1.0 to +1.3), but the average main effect's CI straddles
zero — their apparent benefit doesn't hold up once you average over the whole
combinatorial background, i.e. it's highly context-dependent rather than a
stable, generalizable contribution. By this criterion, K139/K140/K144 are
the "most important" residues in a load-bearing sense — not K124/K127/K130,
despite the latter having the more dramatic single-mutant scores.

The additive model only explains about half the variance (R^2 ~ 0.5), so
there's real epistasis beyond what a per-residue main effect can capture —
consistent with the saturation pattern seen in the faceted view above.

### Does spacing between mutated positions matter?

The additive model above only explains ~50% of variance — the rest is
epistasis. One hypothesis: maybe *how far apart* the mutated positions are
predicts some of that leftover epistasis, beyond just which positions and how
many. Since the single-mutant scores split the 6 positions into two clear
clusters — K124/K127/K130 (positive alone) and K139/K140/K144 (negative
alone) — "spacing" and "does this combo mix the two clusters" turn out to be
almost the same question with only 6 positions in 2 tight groups of 3, so
that's the more useful way to frame it: for each combo, take the model's
residual (actual score minus what the additive main effects alone would
predict) and check whether it depends on staying within one cluster vs.
crossing between them. This is only testable at n=2 and n=3 mutations — from
n=4 on, you're forced to draw from both clusters (only 3 positions per
cluster), so there's no "same-cluster" comparison left to make.

In [ ]:
cluster_A = {124, 127, 130}
cluster_B = {139, 140, 144}

def cluster_group(positions):
    s = set(positions)
    if len(s) < 2:
        return None
    if s <= cluster_A:
        return 'A only (124,127,130)'
    if s <= cluster_B:
        return 'B only (139,140,144)'
    return 'Cross-cluster'

lysala_df['group'] = lysala_df['positions'].apply(cluster_group)
lysala_df['residual'] = lysala_df['D13_log2_median'].values - X_design @ beta
lysala_df['positions_str'] = lysala_df['positions'].apply(lambda ps: ','.join('K' + str(p) for p in ps))

spacing_df = lysala_df[lysala_df['n_mutations'].isin([2, 3])].copy()

spacing_df.groupby(['n_mutations', 'group'])['residual'].agg(['mean', 'count'])

In [ ]:
AQUA = '#1baf7a'
GROUP_ORDER = ['A only (124,127,130)', 'B only (139,140,144)', 'Cross-cluster']

spacing_points = alt.Chart(spacing_df).mark_point(filled=True, size=90, opacity=0.85).encode(
    x=alt.X('group:N', title=None, sort=GROUP_ORDER, axis=alt.Axis(labelAngle=-20)),
    y=alt.Y('D13_log2_median:Q', title='D13 log2 median LFC (raw score)'),
    color=alt.Color(
        'group:N',
        scale=alt.Scale(domain=GROUP_ORDER, range=[BLUE, AQUA, RED]),
        legend=None
    ),
    tooltip=['positions_str:N', alt.Tooltip('D13_log2_median:Q', format='.2f')]
)
spacing_means = alt.Chart(spacing_df).mark_tick(color='#0b0b0b', thickness=2, size=34).encode(
    x=alt.X('group:N', sort=GROUP_ORDER),
    y=alt.Y('mean(D13_log2_median):Q')
)
spacing_zero_ref = alt.Chart(pd.DataFrame({'y': [0]})).mark_rule(color=MUTED, strokeDash=[4, 4]).encode(y='y:Q')

spacing_chart = (spacing_points + spacing_means + spacing_zero_ref).properties(
    width=220, height=280
).facet(
    facet=alt.Facet('n_mutations:O', title='# mutations'),
    columns=2
).properties(
    title='Within-cluster vs cross-cluster combos: raw fitness scores'
).configure(
    background='#fcfcfb'
).configure_axis(
    gridColor='#e1e0d9', domainColor='#c3c2b7', labelColor='#52514e', titleColor='#0b0b0b'
).configure_view(
    strokeWidth=0
).configure_title(
    color='#0b0b0b'
)

spacing_chart

**Reading the chart:** at n=2 (3 same-cluster-A, 3 same-cluster-B, 9
cross-cluster pairs), the raw scores split cleanly: combining two positions
from the beneficial cluster (K124/K127/K130) stays strongly positive (mean
+1.21), combining two from the damaging cluster (K139/K140/K144) is the most
negative group (mean -0.60), and cross-cluster pairs land in between
(mean -0.40) — closer to the damaging cluster's level than to a true midpoint,
consistent with the negative main effects dominating once a damaging-cluster
position is involved. The same ordering holds at n=3, though only 2
same-cluster triples exist there (one per cluster), so treat that facet as
suggestive rather than conclusive.

One thing to keep in mind reading raw scores this way: each group is also
averaging over different position identities (A-only pairs are always some
mix of K124/K127/K130, cross-cluster pairs always include one damaging-cluster
position), so part of what separates the groups is simply which positions
they contain, not purely an interaction/spacing effect. The additive model
above is what isolates the "beyond what these specific positions alone would
predict" component — this raw view answers a simpler, more direct question:
what do same-cluster vs. cross-cluster combos actually score, in the
experiment's own units.

## AAE Mutants

In [ ]:
aae_df = lib_a_final[lib_a_final['oligo_name'].str.contains('AAE')].copy()
aae_df

In [ ]:
aae_position_order = [133, 135, 136]

aae_df['positions'] = aae_df['var_name'].apply(lambda v: [int(p) for p in re.findall(r'[A-Z](\d+)', v)])
aae_df['n_mutations'] = aae_df['positions'].apply(len)

aae_long = aae_df.explode('positions').rename(columns={'positions': 'position'})
aae_long['position'] = aae_long['position'].astype(int)

aae_long

In [ ]:
lines_aae = alt.Chart().mark_line(
    color=BLUE,
    strokeWidth=2,
    point=alt.OverlayMarkDef(color=BLUE, filled=True, size=70, stroke='white', strokeWidth=2)
).encode(
    x=alt.X('position:O', sort=aae_position_order, title='Mutated position'),
    y=alt.Y('D13_log2_median:Q', title='D13 log2 median LFC', scale=alt.Scale(zero=False, padding=10)),
    detail='var_name:N',
    tooltip=['var_name:N', alt.Tooltip('D13_log2_median:Q', format='.2f'), 'n_mutations:O']
)

ref_line_aae = alt.Chart(pd.DataFrame({'y': [DEPLETION_THRESHOLD]})).mark_rule(
    color=MUTED, strokeWidth=1, strokeDash=[4, 4]
).encode(
    y='y:Q'
)

# No AAE combo crosses the depletion threshold (all scores stay above -0.5),
# so the hotness background from the Lys->Ala barbell doesn't apply here --
# just the lines and the reference line.
aae_barbell = alt.layer(lines_aae, ref_line_aae, data=aae_long).properties(
    width=220, height=260
).facet(
    facet=alt.Facet('n_mutations:O', title='# mutations'),
    columns=3
).resolve_scale(
    y='independent'
).properties(
    title='AAE combinations (F133A, D135A, A136E): D13 log2 median fitness by mutated residue'
).configure(
    background='#fcfcfb'
).configure_axis(
    gridColor='#e1e0d9', domainColor='#c3c2b7', labelColor='#52514e', titleColor='#0b0b0b'
).configure_view(
    strokeWidth=0
).configure_title(
    color='#0b0b0b'
)

aae_barbell

## SNVs

In [ ]:
full_sge_dataset=pd.read_excel('./Data/final_tables/supplementary_file_1_BARD1_SGE_final_table.xlsx', sheet_name='scores')

# 'target' is a semicolon-joined list of tiles for variants that fall in a
# tile-overlap region (e.g. 'BARD1_X4A;BARD1_X4B'), not always a single tile
# name — an exact == 'BARD1_X4A' match silently drops all of those overlap
# rows even though they're genuinely BARD1_X4A variants too.
x4a_sge_scores = full_sge_dataset[full_sge_dataset['target'].str.contains('BARD1_X4A', na=False, regex=False)].copy()
x4a_sge_scores=x4a_sge_scores[['consequence', 'pos_id', 'amino_acid_change', 'score', 'functional_consequence']]

print(len(x4a_sge_scores))

idr_snvs_df = lib_a_final[lib_a_final['oligo_name'].str.contains('SNV')].copy()
print(len(idr_snvs_df))
final_snvs_df = pd.merge(idr_snvs_df, x4a_sge_scores, left_on='var_name', right_on='pos_id', how='inner'
                         )

# 3 of the 330 SNVs (all at genomic position 214781488) have no match: that
# exact position isn't annotated as a standalone SNV anywhere in the final
# table at all -- it's only covered there as part of an unrelated 3bp
# deletion call (214781489-214781491). Real data gap, not a merge bug.
print(f'{len(idr_snvs_df) - len(final_snvs_df)} SNVs unmatched: '
      f'{sorted(set(idr_snvs_df["var_name"]) - set(final_snvs_df["var_name"]))}')

final_snvs_df['pos'] = final_snvs_df['pos_id'].transform(lambda x: int(x.split(':')[0]))
final_snvs_df

In [ ]:
palette = [
    '#006616', # dark green,
    '#81B4C7', # dusty blue
    '#ffcd3a', # yellow
    '#6AA84F', # med green
    '#1170AA', # darker blue
    '#CFCFCF' # light gray
]


variant_types = [
    'synonymous_variant',
    'missense_variant',  
    'stop_gained',
    'intron_variant', 
    'splice_site_variant', 
    'splicing_variant',
]

snv_scatter = alt.Chart(final_snvs_df).mark_point().encode(
    x=alt.X('pos:Q',
            scale=alt.Scale(zero=False)
    ),
    y='D13_log2_median:Q',
    tooltip=['pos_id','amino_acid_change', 'D13_log2_median'],
    color=alt.Color('consequence:N',
                    scale=alt.Scale(domain=variant_types,
                                    range=palette)
    ),
    shape='functional_consequence:N'

).properties(
    width = 750
).interactive()

snv_scatter.display()


In [ ]:
corr, _ = stats.pearsonr(final_snvs_df['D13_log2_median'], final_snvs_df['score'])
print(corr)

snv_comparison = alt.Chart(final_snvs_df).mark_circle().encode(
    x='D13_log2_median:Q',
    y='score:Q',
    color=alt.Color('consequence:N',
                    scale=alt.Scale(domain=variant_types,
                                    range=palette)
    ),
    tooltip=['pos_id', 'amino_acid_change']
).interactive()

snv_comparison.display()

# Analyze Library B

## Heatmap

In [ ]:
lib_b_df = lib_b_lfcs[lib_b_lfcs['AA_pos'] != 129].copy()

# One row per position, marking which resultAA cell is actually the wild-type
# residue there, so it can be outlined on the heatmap.
orig_aa_marks = lib_b_df.drop_duplicates(subset=['AA_pos'])[['AA_pos', 'origAA']]

# Explicit extent+step (rather than maxbins/minstep) so every integer position
# gets exactly one bin, with an unambiguous edge for the last one. The
# previous maxbins=25/minstep=1 spec computed only 24 unit bins over a
# domain spanning 24 (147-123), so the last bin ended up inclusive on both
# ends and merged positions 146 and 147 into one column. Keeping this
# quantitative (rather than ordinal) preserves the visual gap at the
# excluded position 129.
bin_spec = alt.Bin(extent=[123, 148], step=1)

lib_b_heatmap_base = alt.Chart(lib_b_df).mark_rect().encode(
    x=alt.X('AA_pos:Q',
            axis=alt.Axis(
                values=list(range(120, 150, 5))
            ),
            scale=alt.Scale(
                domain=[123,148]
            ),
            bin=bin_spec
    ),
    y=alt.Y('resultAA:N'),
    color=alt.Color('D13_log2_median:Q',
                    scale = alt.Scale(
                                  domain = [-2,0],
                                  clamp = True,
                                  reverse=True,
                                  scheme = 'bluepurple'
                              )
                    ),
    tooltip=['AA_pos','origAA','resultAA','D13_log2_median']
)

lib_b_heatmap_orig_box = alt.Chart(orig_aa_marks).mark_rect(
    fill=None, stroke='black', strokeWidth=2
).encode(
    x=alt.X('AA_pos:Q', bin=bin_spec, scale=alt.Scale(domain=[123,148])),
    y=alt.Y('origAA:N')
)

lib_b_heatmap = (lib_b_heatmap_base + lib_b_heatmap_orig_box).properties(
    height=300,
    width=500
).configure_axis(
        grid = False
    ).configure_view(
        stroke = None
    )

lib_b_heatmap.display()

In [ ]:
fullsat_scores_df = full_sat_fitness_scores[full_sat_fitness_scores['AA_pos'] != 129].copy()

fullsat_scores_df.loc[fullsat_scores_df['resultAA']=='X', 'resultAA'] = '*'
order = ['A', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'P', 'Q', 'R', 'S', 'T', 'V', 'W', 'Y', '*',]

orig_aa_marks = fullsat_scores_df.drop_duplicates(subset=['AA_pos'])[['AA_pos', 'origAA']]
orig_aa_marks = orig_aa_marks.rename(columns={'origAA': 'resultAA'})

bin_spec = alt.Bin(extent=[123, 148], step=1)

lib_b_heatmap_base = alt.Chart(fullsat_scores_df).mark_rect().encode(
    x=alt.X('AA_pos:Q',
            axis=alt.Axis(
                values=list(range(120, 150, 5)),
                format='.0f'
            ),
            scale=alt.Scale(
                domain=[123,148]
            ),
            bin=bin_spec
    ),
    y=alt.Y('resultAA:O',
            sort=order),
    color=alt.Color('modelscore:Q',
                    scale = alt.Scale(
                                  domain = [-0.2,0],
                                  clamp = True,
                                  reverse=True,
                                  scheme = 'bluepurple'
                              )
                    ),
    tooltip=['AA_pos','origAA','resultAA','modelscore']
)

lib_b_heatmap_orig_box = alt.Chart(orig_aa_marks).mark_rect(
    fill=None, stroke='black', strokeWidth=2
).encode(
    x=alt.X('AA_pos:Q', bin=bin_spec, scale=alt.Scale(domain=[123,148])),
    y=alt.Y('resultAA:O', sort=order)
)

lib_b_scores_heatmap = (lib_b_heatmap_base + lib_b_heatmap_orig_box).properties(
    height=300,
    width=400
).configure_axis(
        grid = False
    ).configure_view(
        stroke = None
    )

lib_b_scores_heatmap.display()

## Residue Type Analysis for Depleted

In [ ]:
CHARGE_CLASS = {aa: 'positive' for aa in 'KR'}
CHARGE_CLASS.update({aa: 'negative' for aa in 'DE'})
CHARGE_CLASS.update({aa: 'neutral' for aa in 'ACFGHILMNPQSTVWY'})

# Kyte-Doolittle hydrophobicity scale
KYTE_DOOLITTLE = {
    'I': 4.5, 'V': 4.2, 'L': 3.8, 'F': 2.8, 'C': 2.5, 'M': 1.9, 'A': 1.8, 'G': -0.4,
    'T': -0.7, 'S': -0.8, 'W': -0.9, 'Y': -1.3, 'P': -1.6, 'H': -3.2, 'E': -3.5,
    'Q': -3.5, 'D': -3.5, 'N': -3.5, 'K': -3.9, 'R': -4.5,
}

# Residue volume in cubic angstroms (Zamyatnin 1972)
VOLUME = {
    'G': 60.1, 'A': 88.6, 'S': 89.0, 'C': 108.5, 'D': 111.1, 'P': 112.7, 'N': 114.1,
    'T': 116.1, 'E': 138.4, 'V': 140.0, 'Q': 143.8, 'H': 153.2, 'M': 162.9, 'I': 166.7,
    'L': 166.7, 'K': 168.6, 'R': 173.4, 'F': 189.9, 'Y': 193.6, 'W': 227.8,
}

NONPOLAR = set('AVLIMFWPGC')
AROMATIC = set('FWY')

def charge_change(o, r):
    co, cr = CHARGE_CLASS[o], CHARGE_CLASS[r]
    if co == cr:
        return 'No change'
    if {co, cr} == {'positive', 'negative'}:
        return 'Charge reversal'
    return 'Charge gained' if co == 'neutral' else 'Charge lost'

def hydrophobicity_change(o, r, threshold=1.5):
    d = KYTE_DOOLITTLE[r] - KYTE_DOOLITTLE[o]
    if abs(d) < threshold:
        return 'Little change'
    return 'More hydrophobic' if d > 0 else 'More hydrophilic'

def size_change(o, r, threshold=20):
    d = VOLUME[r] - VOLUME[o]
    if abs(d) < threshold:
        return 'Similar size'
    return 'Larger' if d > 0 else 'Smaller'

def polarity_change(o, r):
    po = 'Nonpolar' if o in NONPOLAR else 'Polar'
    pr = 'Nonpolar' if r in NONPOLAR else 'Polar'
    return 'No change' if po == pr else f'{po}->{pr}'

def aromaticity_change(o, r):
    ao, ar = o in AROMATIC, r in AROMATIC
    if ao == ar:
        return 'No change'
    return 'Gained aromatic' if ar else 'Lost aromatic'

# Exclude stop-codon results (no defined biochemical properties) and
# synonymous substitutions (not a "change"). Reuses lib_b_df (already
# excludes the anomalous AA_pos 129) and the DEPLETION_THRESHOLD from the
# Lys->Ala section above.
missense_b = lib_b_df[(lib_b_df['resultAA'] != 'X') & (lib_b_df['origAA'] != lib_b_df['resultAA'])].copy()
missense_b['Charge'] = missense_b.apply(lambda r: charge_change(r['origAA'], r['resultAA']), axis=1)
missense_b['Hydrophobicity'] = missense_b.apply(lambda r: hydrophobicity_change(r['origAA'], r['resultAA']), axis=1)
missense_b['Size'] = missense_b.apply(lambda r: size_change(r['origAA'], r['resultAA']), axis=1)
missense_b['Polarity'] = missense_b.apply(lambda r: polarity_change(r['origAA'], r['resultAA']), axis=1)
missense_b['Aromaticity'] = missense_b.apply(lambda r: aromaticity_change(r['origAA'], r['resultAA']), axis=1)

depleted_b = missense_b[missense_b['D13_log2_median'] < DEPLETION_THRESHOLD].copy()
print(f'{len(depleted_b)} depleted missense variants (of {len(missense_b)} total missense)')

property_cols = ['Charge', 'Hydrophobicity', 'Size', 'Polarity', 'Aromaticity']
depleted_long = depleted_b.melt(
    id_vars=['oligo_name', 'origAA', 'resultAA', 'AA_pos', 'D13_log2_median'],
    value_vars=property_cols, var_name='property', value_name='category'
)

CATEGORY_ORDER = [
    'No change', 'Little change', 'Similar size',
    'Charge lost', 'Charge gained', 'Charge reversal',
    'More hydrophilic', 'More hydrophobic',
    'Smaller', 'Larger',
    'Polar->Nonpolar', 'Nonpolar->Polar',
    'Lost aromatic', 'Gained aromatic',
]

residue_type_box = alt.Chart(depleted_long).mark_boxplot(color=BLUE, size=30, outliers=False).encode(
    x=alt.X('category:N', sort=CATEGORY_ORDER, title=None, axis=alt.Axis(labelAngle=-30)),
    y=alt.Y('D13_log2_median:Q', title='D13 log2 median LFC')
)
residue_type_pts = alt.Chart(depleted_long).mark_point(color=BLUE, opacity=0.4, size=25).encode(
    x=alt.X('category:N', sort=CATEGORY_ORDER),
    y=alt.Y('D13_log2_median:Q'),
    tooltip=['origAA:N', 'resultAA:N', 'AA_pos:Q', alt.Tooltip('D13_log2_median:Q', format='.2f')]
)
residue_type_ref = alt.Chart(pd.DataFrame({'y': [DEPLETION_THRESHOLD]})).mark_rule(
    color=MUTED, strokeDash=[4, 4]
).encode(y='y:Q')

residue_type_chart = (residue_type_box + residue_type_pts + residue_type_ref).properties(
    width=220, height=260
).facet(
    facet=alt.Facet('property:N', title=None, sort=property_cols),
    columns=3
).resolve_scale(
    x='independent'
).properties(
    title='Depleted full-saturation variants: score by type of biochemical change'
).configure(
    background='#fcfcfb'
).configure_axis(
    gridColor='#e1e0d9', domainColor='#c3c2b7', labelColor='#52514e', titleColor='#0b0b0b'
).configure_view(
    strokeWidth=0
).configure_title(
    color='#0b0b0b'
)

residue_type_chart

**Reading the chart:** these box plots (and the count plot below) show the
D13 log2 median score, split by the type of biochemical change, for the 66
depleted (score < -0.5) full-saturation missense variants. No single
property axis cleanly separates out as *the* driver of depletion here —
charge, hydrophobicity, size, polarity, and aromaticity are correlated with
each other (a substitution that loses a charge often also goes from polar to
nonpolar and loses bulk, since these are properties of the same 20 amino
acids, not independent dials), and several of the more extreme-looking
categories turn out to be largely the same handful of substitutions viewed
through different property lenses. Combined with thin sample sizes in some
categories (as low as n=4-9), this is best read as descriptive of what the
depleted set looks like, not as evidence that one particular biochemical
property is *the* cause of depletion.

In [ ]:
counts_bar = alt.Chart(depleted_long).mark_bar(color=BLUE, size=30).encode(
    x=alt.X('category:N', sort=CATEGORY_ORDER, title=None, axis=alt.Axis(labelAngle=-30)),
    y=alt.Y('count():Q', title='# depleted variants')
)
counts_labels = alt.Chart(depleted_long).mark_text(dy=-6, color='#0b0b0b', fontSize=11).encode(
    x=alt.X('category:N', sort=CATEGORY_ORDER),
    y=alt.Y('count():Q'),
    text='count():Q'
)

counts_chart = (counts_bar + counts_labels).properties(width=220, height=220).facet(
    facet=alt.Facet('property:N', title=None, sort=property_cols),
    columns=3
).resolve_scale(
    x='independent'
).properties(
    title='Depleted full-saturation variants: count by type of biochemical change'
).configure(
    background='#fcfcfb'
).configure_axis(
    gridColor='#e1e0d9', domainColor='#c3c2b7', labelColor='#52514e', titleColor='#0b0b0b'
).configure_view(
    strokeWidth=0
).configure_title(
    color='#0b0b0b'
)

counts_chart

## SNV vs Full Saturation: Missense Score Comparison

In [ ]:
# Include synonymous variants alongside missense (still excluding nonsense/
# stop, which is categorically different and uniformly extreme on both sides).
coding_snv = final_snvs_df[final_snvs_df['consequence'].isin(['missense_variant', 'synonymous_variant'])].copy()

coding_b = lib_b_df[lib_b_df['resultAA'] != 'X'].copy()
coding_b['amino_acid_change'] = (
    coding_b['origAA'] + coding_b['AA_pos'].astype(int).astype(str) + coding_b['resultAA']
)

snv_vs_fullsat = pd.merge(
    coding_snv[['amino_acid_change', 'D13_log2_median', 'consequence']].rename(columns={'D13_log2_median': 'snv_score'}),
    coding_b[['amino_acid_change', 'D13_log2_median']].rename(columns={'D13_log2_median': 'fullsat_score'}),
    on='amino_acid_change', how='inner'
)

r = snv_vs_fullsat['snv_score'].corr(snv_vs_fullsat['fullsat_score'])

def depletion_category(row):
    snv_dep = row['snv_score'] <= DEPLETION_THRESHOLD
    fullsat_dep = row['fullsat_score'] <= DEPLETION_THRESHOLD
    if snv_dep and fullsat_dep:
        return 'Depleted in both'
    if snv_dep:
        return 'Depleted in SNV only'
    if fullsat_dep:
        return 'Depleted in full-sat only'
    return 'Not depleted'

snv_vs_fullsat['category'] = snv_vs_fullsat.apply(depletion_category, axis=1)

SNV_FULLSAT_CATEGORY_ORDER = ['Not depleted', 'Depleted in SNV only', 'Depleted in full-sat only', 'Depleted in both']
SNV_FULLSAT_CATEGORY_COLORS = [MUTED, "#a19f0c", '#1baf7a', RED]

snv_fullsat_points = alt.Chart(snv_vs_fullsat).mark_point(filled=True, opacity=0.75, size=55).encode(
    x=alt.X('snv_score:Q', title='SNV-encoded score (D13 log2 median)'),
    y=alt.Y('fullsat_score:Q', title='Full-saturation score (D13 log2 median)'),
    shape=alt.Shape('consequence:N', title=None),
    color=alt.Color(
        'category:N',
        scale=alt.Scale(domain=SNV_FULLSAT_CATEGORY_ORDER, range=SNV_FULLSAT_CATEGORY_COLORS),
        legend=alt.Legend(title=None)
    ),
    tooltip=[
        'amino_acid_change:N',
        'consequence:N',
        alt.Tooltip('snv_score:Q', format='.2f'),
        alt.Tooltip('fullsat_score:Q', format='.2f'),
        'category:N'
    ]
)

snv_fullsat_lo = min(snv_vs_fullsat['snv_score'].min(), snv_vs_fullsat['fullsat_score'].min())
snv_fullsat_hi = max(snv_vs_fullsat['snv_score'].max(), snv_vs_fullsat['fullsat_score'].max())
snv_fullsat_identity = alt.Chart(pd.DataFrame({'x': [snv_fullsat_lo, snv_fullsat_hi], 'y': [snv_fullsat_lo, snv_fullsat_hi]})).mark_line(
    color=MUTED, strokeDash=[4, 4]
).encode(x='x:Q', y='y:Q')

snv_fullsat_threshold = alt.Chart(pd.DataFrame({'t': [DEPLETION_THRESHOLD]})).mark_rule(color=MUTED, strokeDash=[2, 2], opacity=0.6)
snv_fullsat_vline = snv_fullsat_threshold.encode(x='t:Q')
snv_fullsat_hline = snv_fullsat_threshold.encode(y='t:Q')

snv_vs_fullsat_chart = (snv_fullsat_identity + snv_fullsat_vline + snv_fullsat_hline + snv_fullsat_points).properties(
    width=440, height=440,
    title=f'SNV vs full-saturation scores, missense + synonymous (n={len(snv_vs_fullsat)}, r={r:.2f})'
).configure(
    background='#fcfcfb'
).configure_axis(
    gridColor='#e1e0d9', domainColor='#c3c2b7', labelColor='#52514e', titleColor='#0b0b0b'
).configure_view(
    strokeWidth=0
).configure_title(
    color='#0b0b0b'
)

snv_vs_fullsat_chart